# 08 — SLR-AR Evaluation — Spectral Collapse Mitigation (ViT-Base/16, CRC Histology)

Tests **SLR-AR** (Spectral-Laplacian Regularized Attention Rollout) against standard
Attention Rollout on a trained **ViT-Base/16** checkpoint, CRC-VAL-HE-7K test set (9 classes).

`slr_ar.py` and `eval_slrar.py` (this repo, `notebooks/`) implement the method and its
reference evaluation script; this notebook reruns the same logic interactively on Kaggle
with inline diagnostics and figures.

## Ablation grid (Table 3 of the paper)

| Variant | Spectral norm bounding (SNB) | Graph-theoretic smoothing (GTS) |
|---|---|---|
| `rollout` | ✗ | ✗ | standard Abnar & Zuidema rollout (baseline) |
| `snb` | ✓ | ✗ |
| `gts` | ✗ | ✓ |
| `slrar` | ✓ | ✓ | full method |

## Metrics

- **Spectral Stability Index (SSI)** = \|λ₂\|/\|λ₁\| of the aggregated rollout matrix — label-free,
  cheapest go/no-go signal for spectral collapse. Paper reference (ImageNet, ViT-B/16):
  standard rollout ≈ 0.02, full SLR-AR ≈ 0.24.
- **Insertion / Deletion AUC** (patch-level, Gaussian-blur baseline) — faithfulness to the
  model's own prediction.
- **Qualitative overlays** and **SSI-vs-depth** collapse curves.

**Restricted to `vit_base`**: `slr_ar.py`'s `AttentionCatcher` and `cls_attribution_map`
assume a single-CLS-token, non-windowed `model.blocks` layout. DeiT-Base has an extra
`[DIST]` token (breaks the 14×14 reshape), Swin-Base has no global CLS attention, and
DINOv2 nests blocks under `backbone.blocks`. Per CLAUDE.md, ViT-Base/16 is the obligatory
SOTA ViT anchor model, so this is also the correct default.

**Edit only Cell 2** (`USER CONFIG`).

**Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q PyDrive2
!pip install -q scikit-learn

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

MODEL_NAME = "vit_base"   # slr_ar.py targets ViT-Base/16 CLS-token readout only — do not change

# Google Drive folder containing the vit_base checkpoint (and where results are uploaded)
DRIVE_FOLDER_ID = "1FqMB8TwwuBadOxhnqF4IgLFAw9e-nwTU"  # vit_base <-- update as needed

# Dataset subset
N_IMAGES_PER_CLASS = 30
BATCH_SIZE          = 8
NUM_WORKERS         = 2
IMAGE_SIZE          = 224
SEED                = 42

# SLR-AR hyperparameters (paper defaults, Sec 5.1)
TAU            = 0.95        # spectral norm threshold
K              = 5           # power iteration steps
M              = 8           # Taylor expansion order
RESIDUAL_ALPHA = 0.5         # P = alpha*A + (1-alpha)*I, then row-normalise
SMOOTHING      = "applied"   # 'paper' -> Q = exp(-beta L)  |  'applied' -> Q = exp(-beta L) @ P_tilde
BETA_MODE      = "adaptive"  # 'adaptive' -> beta = 1/lambda2  |  'fixed'
EXACT_EXP      = True        # REQUIRED: True -> torch.linalg.matrix_exp (stable).
                             # With False the order-M Taylor series of exp(-beta L)
                             # DIVERGES once beta*lambda_max(L) exceeds ~1, which the
                             # adaptive beta=1/lambda2 guarantees on collapsed attention
                             # (|Q|_2 blows up to ~180, SSI -> 0). N=197 makes the exact
                             # exponential cheap, so there is no reason to use Taylor here.
RENORM_ROWS    = False       # re-row-normalise Q (breaks the tau bound)

VARIANTS = ["rollout", "snb", "gts", "slrar"]   # ablation grid (Table 3 of the paper)

# Faithfulness (patch-level insertion/deletion, Gaussian-blur baseline)
FAITH_N_STEPS = 49
N_VIZ         = 6   # number of images shown in the qualitative overlay grid

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"
CKPT_LOCAL        = "/kaggle/input/models/youssefnouiouar1/crt-vit-base/pytorch/default/1/vit_base_patch16_best.pth"  

In [ ]:
import csv
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm

assert MODEL_NAME == "vit_base", (
    "slr_ar.py assumes a single-CLS-token, non-windowed ViT-B/16 — "
    "see the restriction note in the header cell."
)

for p in (PROJECT_ROOT, f"{PROJECT_ROOT}/notebooks"):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed
from slr_ar import (
    AttentionCatcher, SLRARConfig, build_rollout, cls_attribution_map,
    get_attentions, spectral_stability_index, ssi_by_depth,
)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

GRID  = IMAGE_SIZE // 16   # ViT-B/16 patch grid (14 for 224px)
PATCH = 16

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/slrar/{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)

print(f"Device  : {DEVICE}")
print(f"Classes : {CLASS_NAMES}")
print(f"Grid    : {GRID}x{GRID} patches")
print(f"Output  : {SAVE_DIR}")
print(f"Checkpoint path: {CKPT_LOCAL}")

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────

def load_state(model: nn.Module, ckpt_path: str) -> dict:
    """Load checkpoint weights; return metadata dict."""
    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    if isinstance(state, dict) and "state_dict" in state:
        cleaned = {k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}
        model.load_state_dict(cleaned, strict=False)
        return {}
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(cleaned, strict=False)
    return {}


model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=NUM_CLASSES,
    global_pool="token",   # CLS-token readout, required by cls_attribution_map
    drop_path_rate=0.0,    # inference
)
meta = load_state(model, CKPT_LOCAL)
model = model.to(DEVICE).eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model            : {MODEL_NAME}")
print(f"Parameters       : {total_params:,}")
print(f"Checkpoint epoch : {meta.get('epoch', '?')}")
print(f"Checkpoint acc   : {meta.get('val_acc', '?')}")

In [ ]:
# ── Dataset & test subset ──────────────────────────────────────────────────
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.25,
    random_state=SEED,
)

test_dataset = CRCHistologyDataset(
    split="test",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=True,
)

from collections import defaultdict

class_counts: dict[int, int] = defaultdict(int)
subset_indices: list[int] = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_IMAGES_PER_CLASS:
        subset_indices.append(idx)
        class_counts[lbl] += 1
    if all(class_counts[c] >= N_IMAGES_PER_CLASS for c in range(NUM_CLASSES)):
        break

eval_dataset = Subset(test_dataset, subset_indices)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Images per class:")
for ci, cn in enumerate(CLASS_NAMES):
    print(f"  {cn}: {class_counts[ci]}")
print(f"Total: {len(subset_indices)} | Batches: {len(eval_loader)}")

In [ ]:
# ── SLR-AR config & attention capture ──────────────────────────────────────
cfg = SLRARConfig(
    tau=TAU,
    K=K,
    M=M,
    residual_alpha=RESIDUAL_ALPHA,
    smoothing=SMOOTHING,
    beta_mode=BETA_MODE,
    exact_exp=EXACT_EXP,
    renorm_rows=RENORM_ROWS,
)
print(cfg)

catcher = AttentionCatcher(model)
print(f"AttentionCatcher registered on {len(catcher.handles)} blocks")

In [ ]:
# ── Faithfulness metric (patch-level insertion/deletion, blur baseline) ────

def _blur(x: torch.Tensor, sigma_px: int = 11) -> torch.Tensor:
    """Gaussian-blurred baseline: a far more plausible 'absent' reference for
    H&E patches than black/gray, which create out-of-distribution artefacts."""
    k = sigma_px if sigma_px % 2 == 1 else sigma_px + 1
    coords = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g = torch.exp(-(coords ** 2) / (2 * (k / 3.0) ** 2))
    g = (g / g.sum()).view(1, 1, 1, k)
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2,) * 2 + (0, 0), mode="reflect"), g.expand(C, 1, 1, k), groups=C)
    x = F.conv2d(F.pad(x, (0, 0) + (k // 2,) * 2, mode="reflect"), g.transpose(-1, -2).expand(C, 1, k, 1), groups=C)
    return x


@torch.no_grad()
def insertion_deletion_auc(model, x, sal, target, mode, n_steps=FAITH_N_STEPS, patch=PATCH, grid=GRID, fwd_bs=32):
    """AUC over patch-level insertion or deletion, one image at a time.

    x   : (1, 3, H, W) normalised input
    sal : (1, grid, grid) saliency
    target: class index the curve is scored against (the model's own prediction,
            not the label — faithfulness measures agreement with the model).
    """
    device = x.device
    order = torch.argsort(sal.reshape(-1), descending=True)
    n_patch = grid * grid
    per_step = max(1, n_patch // n_steps)
    steps = list(range(0, n_patch + 1, per_step))
    if steps[-1] != n_patch:
        steps.append(n_patch)

    base = _blur(x) if mode == "insertion" else torch.zeros_like(x)
    start, fill = (base, x) if mode == "insertion" else (x, base)

    masks = []
    for s in steps:
        m = torch.zeros(n_patch, device=device)
        m[order[:s]] = 1.0
        m = m.reshape(1, 1, grid, grid)
        masks.append(F.interpolate(m, scale_factor=patch, mode="nearest"))
    masks = torch.cat(masks, dim=0)

    imgs = start * (1 - masks) + fill * masks
    probs = []
    for i in range(0, imgs.shape[0], fwd_bs):
        logits = model(imgs[i:i + fwd_bs])
        probs.append(logits.softmax(-1)[:, target])
    probs = torch.cat(probs).cpu().numpy()
    frac = np.array(steps) / n_patch
    return float(np.trapz(probs, frac))


def overlay_heatmap(image_chw: torch.Tensor, saliency_hw: np.ndarray):
    """Unnormalise the image and upsample the saliency map for overlay plotting."""
    img = image_chw.detach().cpu().numpy().transpose(1, 2, 0)
    mean = IMAGENET_MEAN.view(3).numpy()
    std = IMAGENET_STD.view(3).numpy()
    img = np.clip(img * std + mean, 0.0, 1.0)
    sal = saliency_hw.astype(np.float32)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    sal_up = F.interpolate(
        torch.from_numpy(sal)[None, None], size=img.shape[:2], mode="bilinear", align_corners=False
    )[0, 0].numpy()
    return img, sal_up


print("Evaluation helpers ready")

In [ ]:
# ── Main evaluation loop — SSI, insertion/deletion AUC per variant ─────────
acc = {v: {"ssi": [], "ins": [], "del": [], "qnorm": []} for v in VARIANTS}
depth_curves: dict[str, torch.Tensor] = {}
viz_bank: list[tuple[str, torch.Tensor]] = []
viz_x, viz_y = None, None
per_image_rows: list[dict] = []

n_correct = 0
n_total = 0

for bi, (images, labels, image_ids) in enumerate(tqdm(eval_loader, desc="SLR-AR eval")):
    images = images.to(DEVICE, non_blocking=True)
    catcher.resume()
    logits, attns = get_attentions(model, images, catcher)
    catcher.pause()   # faithfulness forward passes below must NOT be captured —
                       # otherwise attn_drop's hook leaks a (B,heads,N,N) tensor per
                       # layer on every masked-image forward and GPU memory explodes
    preds = logits.argmax(-1)
    n_correct += int((preds.cpu() == labels).sum())
    n_total += images.shape[0]

    for v in VARIANTS:
        R, diags = build_rollout(attns, v, cfg)
        sal = cls_attribution_map(R, grid=GRID)
        ssi_batch = spectral_stability_index(R).cpu()
        qnorm_batch = torch.stack(diags["q_norm"]).mean(0).cpu()   # mean over depth, per image
        acc[v]["ssi"].append(ssi_batch)
        acc[v]["qnorm"].append(qnorm_batch)

        for i in range(images.shape[0]):
            t = int(preds[i])
            ins = insertion_deletion_auc(model, images[i:i + 1], sal[i:i + 1], t, "insertion")
            dele = insertion_deletion_auc(model, images[i:i + 1], sal[i:i + 1], t, "deletion")
            acc[v]["ins"].append(ins)
            acc[v]["del"].append(dele)
            per_image_rows.append({
                "image_id": image_ids[i], "label_idx": int(labels[i]),
                "label_name": CLASS_NAMES[int(labels[i])],
                "pred_idx": t, "pred_name": CLASS_NAMES[t], "variant": v,
                "ssi": float(ssi_batch[i]), "insertion_auc": ins, "deletion_auc": dele,
                "faithfulness": ins - dele, "q_norm": float(qnorm_batch[i]),
            })

        if bi == 0:
            depth_curves[v] = ssi_by_depth(attns, v, cfg)
            if len(viz_bank) < len(VARIANTS):
                viz_bank.append((v, sal[:N_VIZ].detach().cpu()))

    if bi == 0:
        viz_x = images[:N_VIZ].detach().cpu()
        viz_y = [CLASS_NAMES[int(c)] for c in labels[:N_VIZ]]

catcher.remove()
model_accuracy = n_correct / n_total
print(f"Model accuracy on subset: {model_accuracy:.4f} ({n_correct}/{n_total})")

In [ ]:
# ── Summary table & save ───────────────────────────────────────────────────
summary = {}
print("=" * 78)
print(f"{'Variant':<10}{'SSI':>10}{'Insertion':>12}{'Deletion':>12}{'Faithfulness':>14}{'mean|Q|_2':>12}")
print("-" * 78)
for v in VARIANTS:
    ssi = torch.cat(acc[v]["ssi"]).mean().item()
    ins = float(np.mean(acc[v]["ins"]))
    dele = float(np.mean(acc[v]["del"]))
    qn = torch.cat(acc[v]["qnorm"]).mean().item()
    summary[v] = {
        "ssi": ssi, "insertion_auc": ins, "deletion_auc": dele,
        "faithfulness": ins - dele, "mean_q_spectral_norm": qn,
    }
    print(f"{v:<10}{ssi:>10.4f}{ins:>12.4f}{dele:>12.4f}{ins - dele:>14.4f}{qn:>12.4f}")
print("=" * 78)
print("Higher SSI = less spectral collapse | Higher insertion, lower deletion = more faithful")

with open(SAVE_DIR / "summary.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "model_accuracy": model_accuracy,
        "n_images": n_total,
        "config": {
            "tau": TAU, "K": K, "M": M, "residual_alpha": RESIDUAL_ALPHA,
            "smoothing": SMOOTHING, "beta_mode": BETA_MODE,
            "exact_exp": EXACT_EXP, "renorm_rows": RENORM_ROWS,
        },
        "results": summary,
    }, f, indent=2)

per_image_csv = SAVE_DIR / "per_image_results.csv"
with open(per_image_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(per_image_rows[0].keys()))
    w.writeheader()
    w.writerows(per_image_rows)

print(f"Saved: {SAVE_DIR / 'summary.json'}")
print(f"Saved: {per_image_csv}")

In [ ]:
# ── Figures: SSI-vs-depth collapse curve, qualitative overlays ─────────────

# SSI vs depth (Figure 4 style)
fig, ax = plt.subplots(figsize=(6, 4))
for v in VARIANTS:
    c = depth_curves[v].mean(1).numpy()
    ax.plot(range(1, len(c) + 1), c, marker="o", label=v)
ax.set_xlabel("Transformer depth (layer)")
ax.set_ylabel("Spectral Stability Index  |\u03bb\u2082|/|\u03bb\u2081|")
ax.set_yscale("log")
ax.set_title(f"{MODEL_NAME} \u2014 SSI vs depth")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(SAVE_DIR / "ssi_by_depth.png", dpi=150)
plt.show()

# Qualitative overlays: input + one column per variant
n = viz_x.shape[0]
fig, axes = plt.subplots(n, len(VARIANTS) + 1, figsize=(3.2 * (len(VARIANTS) + 1), 3.2 * n))
axes = np.atleast_2d(axes)
mean = IMAGENET_MEAN.view(3).numpy()
std = IMAGENET_STD.view(3).numpy()
for r in range(n):
    img = np.clip(viz_x[r].permute(1, 2, 0).numpy() * std + mean, 0, 1)
    axes[r, 0].imshow(img)
    axes[r, 0].set_ylabel(viz_y[r], fontsize=9, rotation=0, labelpad=40, va="center")
    if r == 0:
        axes[r, 0].set_title("input")
    for c, (v, sal) in enumerate(viz_bank):
        img_u, sal_up = overlay_heatmap(viz_x[r], sal[r].numpy())
        axes[r, c + 1].imshow(img_u)
        axes[r, c + 1].imshow(sal_up, cmap="jet", alpha=0.5)
        if r == 0:
            axes[r, c + 1].set_title(v)
    for a in axes[r]:
        a.set_xticks([])
        a.set_yticks([])
fig.tight_layout()
fig.savefig(SAVE_DIR / "qualitative.png", dpi=150)
plt.show()

print(f"Saved: {SAVE_DIR / 'ssi_by_depth.png'}")
print(f"Saved: {SAVE_DIR / 'qualitative.png'}")

## Optional — hyperparameter sweep (SSI only)

Run this only if the main table shows SLR-AR not beating plain `rollout` on SSI, to
search for a configuration where it does. SSI is label-free and needs only the captured
attention maps — **no** insertion/deletion forward passes — so a full grid is cheap.

The sweep fixes `exact_exp=True` (never trust the Taylor path, see the config cell) and
varies:
- `beta_clamp_hi` — upper clamp on the adaptive `beta = 1/lambda2`. A smaller cap means
  gentler diffusion; the paper's implicit cap (20) is aggressive on collapsed attention.
- `smoothing` — `applied` (`Q = exp(-beta L) @ P_tilde`) vs `paper` (`Q = exp(-beta L)`).
- `renorm_rows` — re-row-normalise `Q` (restores row-stochasticity the paper claims but
  the construction does not actually guarantee).

`delta = SSI(slrar) - SSI(rollout)`: **positive means SLR-AR reduces spectral collapse.**

In [ ]:
# ── Hyperparameter sweep (SSI-based, fast) ─────────────────────────────────
import itertools

SWEEP_GRID = {
    "beta_clamp_hi": [2.0, 5.0, 20.0],   # upper clamp on beta = 1/lambda2
    "smoothing"    : ["applied", "paper"],
    "renorm_rows"  : [False, True],
}
SWEEP_MAX_BATCHES = None   # None = all batches; set an int to subsample for speed


def make_sweep_cfg(beta_clamp_hi, smoothing, renorm_rows):
    return SLRARConfig(
        tau=TAU, K=K, M=M, residual_alpha=RESIDUAL_ALPHA,
        smoothing=smoothing, beta_mode="adaptive",
        beta_clamp=(0.1, beta_clamp_hi),
        exact_exp=True,          # always exact in the sweep — Taylor diverges here
        renorm_rows=renorm_rows,
    )


keys = list(SWEEP_GRID.keys())
combos = list(itertools.product(*SWEEP_GRID.values()))
base_cfg = SLRARConfig(tau=TAU, K=K, M=M, residual_alpha=RESIDUAL_ALPHA)  # rollout ignores smoothing/beta
print(f"Sweeping {len(combos)} configs (rollout baseline computed once per batch) ...")

sweep_catcher = AttentionCatcher(model)
rollout_ssi_list = []
slrar_ssi = {ci: [] for ci in range(len(combos))}

for bi, (images, labels, image_ids) in enumerate(tqdm(eval_loader, desc="sweep")):
    if SWEEP_MAX_BATCHES is not None and bi >= SWEEP_MAX_BATCHES:
        break
    images = images.to(DEVICE, non_blocking=True)
    sweep_catcher.resume()
    _, attns = get_attentions(model, images, sweep_catcher)
    sweep_catcher.pause()

    R0, _ = build_rollout(attns, "rollout", base_cfg)
    rollout_ssi_list.append(spectral_stability_index(R0).cpu())

    for ci, combo in enumerate(combos):
        Rs, _ = build_rollout(attns, "slrar", make_sweep_cfg(**dict(zip(keys, combo))))
        slrar_ssi[ci].append(spectral_stability_index(Rs).cpu())

sweep_catcher.remove()

roll = torch.cat(rollout_ssi_list).mean().item()
rows = []
for ci, combo in enumerate(combos):
    d = dict(zip(keys, combo))
    slr = torch.cat(slrar_ssi[ci]).mean().item()
    rows.append({**d, "ssi_rollout": roll, "ssi_slrar": slr, "delta": slr - roll})
rows.sort(key=lambda r: r["delta"], reverse=True)

print(f"\n{'beta_hi':>8}{'smoothing':>10}{'renorm':>8}{'SSI_roll':>10}{'SSI_slrar':>11}{'delta':>10}")
print("-" * 57)
for r in rows:
    print(f"{r['beta_clamp_hi']:>8}{r['smoothing']:>10}{str(r['renorm_rows']):>8}"
          f"{r['ssi_rollout']:>10.4f}{r['ssi_slrar']:>11.4f}{r['delta']:>+10.4f}")
best = rows[0]
print(f"\nBest delta = {best['delta']:+.4f}  "
      f"(beta_hi={best['beta_clamp_hi']}, smoothing={best['smoothing']}, renorm={best['renorm_rows']})")
print("Positive delta = SLR-AR reduces spectral collapse vs plain rollout.")

sweep_csv = SAVE_DIR / "sweep_ssi.csv"
with open(sweep_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f"Saved: {sweep_csv}")

In [ ]:
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID  # same folder as checkpoint; update if needed

files_to_upload = [
    SAVE_DIR / "summary.json",
    SAVE_DIR / "per_image_results.csv",
    SAVE_DIR / "ssi_by_depth.png",
    SAVE_DIR / "qualitative.png",
    SAVE_DIR / "sweep_ssi.csv",   # only exists if the optional sweep cell was run
]

for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}")
        continue
    drive_file = drive.CreateFile({
        "title": f"slrar_{MODEL_NAME}_{fpath.name}",
        "parents": [{"id": RESULTS_DRIVE_FOLDER}],
    })
    drive_file.SetContentFile(str(fpath))
    drive_file.Upload()
    print(f"  Uploaded: slrar_{MODEL_NAME}_{fpath.name}")

print("Done.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")